### Proyecto 2do Bimenstre - Informe Técnico

**Integrantes:**    Hernández Mark & Jiménez Yasid

**Materia:**        Recuperación de la Información

**Docente:**        Carrera Iván 

### Librerías y configuraciones

In [12]:
import pandas as pd
import json
import os
from pathlib import Path
from IPython.display import display
import numpy as np
import math
import logging
import warnings

# 1. Silenciar las advertencias nativas de Python (el texto suelto "Warning:...")
warnings.filterwarnings("ignore")
# 2. Subir el nivel de los módulos ruidosos para que SOLO muestren errores críticos
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)
# 3. Asegurarnos de que tu propio módulo SÍ pueda imprimir sus mensajes (INFO)
logging.getLogger("src.a_corpus_preparation").setLevel(logging.INFO)
# 4. Bloquear los logs de Gemini y de tu propio pipeline de reintentos
logging.getLogger("google_genai.models").setLevel(logging.CRITICAL)
logging.getLogger("src.d_rag_pipeline").setLevel(logging.CRITICAL)

### 1. Descripción del Corpus Multimodal
El sistema integra dos datasets de Hugging Face:
* **ESCI:** Proporciona consultas reales, catálogo textual y juicios de relevancia.
* **SQID:** Aporta la capa visual mediante URLs de imágenes.

**a) Preparación del corpus:** El script `src/a_corpus_preparation.py` limpia el texto, verifica las URLs, descarga las imágenes localmente y empaqueta el corpus unificado.

In [13]:
import src.a_corpus_preparation as corpusPrep

print("▶ Verificación del Literal a)")
examples, products = corpusPrep.load_esci_subset()
image_urls = corpusPrep.load_image_urls(set(products["product_id"]))
print("📊 Dimensiones de los datos extraídos:")
print(f"- Consultas (examples): {examples.shape}")
print(f"- Productos (products): {products.shape}")
print(f"- URLs de imágenes validadas: {image_urls.shape}")

# Generamos el corpus final y mostramos solo 2 registros truncados para ahorrar espacio
documents = corpusPrep.build_multimodal_corpus(products, image_urls)
df_documents = pd.DataFrame(documents)

print("\nVista previa del Corpus Multimodal (Truncado):")
display(df_documents[['doc_id', 'product_title', 'image_url']].head(2))
# Guardamos el corpus y las consultas con sus qrels en disco
valid_ids = {d.product_id for d in documents}
corpusPrep.save_corpus(documents)
corpusPrep.save_queries_and_qrels(examples, valid_ids)
# Comprobamos que el archivo del corpus se haya guardado correctamente
corpus_path = Path("data/corpus/corpus.jsonl")
if corpus_path.exists():
    print(f"✅ Archivo del corpus encontrado exitosamente en: {corpus_path}")
    
    # Leemos solo las 5 primeras líneas para demostrar la limpieza y estructura
    muestras = []
    with open(corpus_path, "r", encoding="utf-8") as f:
        for _ in range(5):
            linea = f.readline()
            if linea:
                muestras.append(json.loads(linea))

2026-07-22 08:13:48,401 [src.a_corpus_preparation] Cargando juicios de relevancia ESCI (split=test)...


▶ Verificación del Literal a)


2026-07-22 08:13:50,546 [src.a_corpus_preparation] Cargando catálogo de productos ESCI...
2026-07-22 08:13:54,141 [src.a_corpus_preparation] Muestra final: 40 consultas, 300 pares (query, producto), 245 productos.
2026-07-22 08:13:54,146 [src.a_corpus_preparation] Cargando URLs de imagen (SQID)...
2026-07-22 08:13:55,098 [src.a_corpus_preparation] Imágenes disponibles para 106/245 productos.


📊 Dimensiones de los datos extraídos:
- Consultas (examples): (300, 10)
- Productos (products): (245, 9)
- URLs de imágenes validadas: (106, 2)


Procesando corpus: 100%|██████████| 245/245 [00:00<00:00, 3379.30it/s]


Vista previa del Corpus Multimodal (Truncado):


,doc_id,product_title,image_url
0,doc_B016J7E3PA,"3 Pair of Classic ""The Intellect"" Full Reading...",https://m.media-amazon.com/images/I/6138IlqjAv...
1,doc_B01CZN30J2,Eyekepper Quality Spring Hinges Wood Temples O...,https://m.media-amazon.com/images/I/51Ra27wzlS...


2026-07-22 08:13:55,223 [src.a_corpus_preparation] Corpus guardado en C:\Users\mark_\Documents\ProyectoIIB-IR\ProyectoIIB\data\corpus\corpus.jsonl (245 documentos).
2026-07-22 08:13:55,242 [src.a_corpus_preparation] Guardadas 39 consultas y qrels para 39 consultas en C:\Users\mark_\Documents\ProyectoIIB-IR\ProyectoIIB\data\qrels\queries.json / C:\Users\mark_\Documents\ProyectoIIB-IR\ProyectoIIB\data\qrels\qrels.json.


✅ Archivo del corpus encontrado exitosamente en: data\corpus\corpus.jsonl


### 2. Arquitectura General y Base Vectorial

**b) Construcción de representaciones vectoriales:** Utilizamos `ClipEmbedder` (modelo CLIP) para fusionar el texto y la imagen de cada producto en un vector único, aplicando normalización L2 para comparaciones equitativas.

**c) Base de datos vectorial:** El almacenamiento se gestiona con ChromaDB (`VectorStore`), configurado espacialmente para medir distancias mediante similitud del coseno (`hnsw:space: cosine`). A continuación, indexamos los vectores y sus metadatos.

In [14]:
from src.b_embeddings import ClipEmbedder
from src.c_vector_store import VectorStore

print("Generación de Embeddings CLIP (Literal b)")
print("Cargando modelo multimodal CLIP en memoria...")
embedder = ClipEmbedder()

textos = [row['text'] for row in muestras]
rutas_img = [row['image_path'] for row in muestras]

print(f"Generando vectores fusionados (texto + imagen) para {len(textos)} documentos...")
# El método embed_documents procesa el texto, abre la imagen con PIL y los fusiona
embeddings = embedder.embed_documents(textos, rutas_img)

print("\n✅ Embeddings generados exitosamente.")
print(f"Forma del tensor matemático (Shape): {embeddings.shape} -> ({len(textos)} documentos, {embeddings.shape[1]} dimensiones)")

# Comprobación de la normalización L2
norma = np.linalg.norm(embeddings[0])
print(f"Norma L2 del primer vector (debe ser aprox. 1.0): {norma:.4f}")

print("Indexación en Base de Datos Vectorial (Literal c)")
# Instanciamos ChromaDB en una colección de prueba para no alterar el index principal
store = VectorStore(collection_name="demo_informe_epn")
store.reset() # Vaciamos la colección por si se ejecuta varias veces

# Preparamos las listas requeridas por ChromaDB
doc_ids = [row['doc_id'] for row in muestras]
# ChromaDB no acepta nulos en metadatos, por lo que reemplazamos None con ""
metadatos = [{"product_title": row['product_title'], "image_path": row['image_path'] or ""} for row in muestras]

print("Insertando documentos en la colección espacial...")
store.index_documents(doc_ids, embeddings, textos, metadatos)
print(f"✅ Total de documentos indexados: {store.count()}")

print("\n🔍 Inspección profunda del motor ChromaDB (Peek):")
# Extraemos 1 registro directamente desde el motor para verificar la estructura interna
sample = store.collection.peek(1)
if sample and sample['ids']:
    df_peek = pd.DataFrame([{
        "ID_Chroma": sample['ids'][0],
        "Dimensiones_Vector": len(sample['embeddings'][0]),
        "Título_Indexado (Metadata)": sample['metadatas'][0].get('product_title', '')[:45] + "...",
        "Similitud_Configurada": "Coseno"
    }])
    print(df_peek.to_dict('records'))

2026-07-22 08:14:15,989 [src.b_embeddings] Cargando modelo CLIP 'openai/clip-vit-base-patch32' en cpu...


Generación de Embeddings CLIP (Literal b)
Cargando modelo multimodal CLIP en memoria...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 6467.58it/s]


Generando vectores fusionados (texto + imagen) para 5 documentos...

✅ Embeddings generados exitosamente.
Forma del tensor matemático (Shape): (5, 512) -> (5 documentos, 512 dimensiones)
Norma L2 del primer vector (debe ser aprox. 1.0): 1.0000
Indexación en Base de Datos Vectorial (Literal c)


2026-07-22 08:14:23,319 [src.c_vector_store] Indexados 5 documentos en la colección 'multimodal_corpus'.


Insertando documentos en la colección espacial...
✅ Total de documentos indexados: 775

🔍 Inspección profunda del motor ChromaDB (Peek):
[{'ID_Chroma': 'doc_B06Y2F3NSS', 'Dimensiones_Vector': 512, 'Título_Indexado (Metadata)': 'TOKYOMILK Eau De Parfum, Song Of The Siren, 1...', 'Similitud_Configurada': 'Coseno'}]


### 3. Pipeline de Recuperación y Generación

**d) Sistema RAG:** El módulo `RAGPipeline` orquesta el flujo en tres fases fundamentales sin romper la modularidad:
1. **Retrieve:** Búsqueda vectorial de candidatos en ChromaDB.
2. **Build Context:** Estructuración de la información para evitar alucinaciones.
3. **Generate:** Respuesta generada por Gemini.

In [15]:
from src.d_rag_pipeline import RAGPipeline

print("Sistema RAG (Literal d)")
# Instanciamos el pipeline inyectando el store y embedder generados en la sección 2.
pipeline = RAGPipeline(vector_store=store, embedder=embedder)
# Paso 1: Definir la consulta
query_demo = "reading sunglasses for women"
print(f"👤 Consulta del usuario: '{query_demo}'")
# Paso 2: Búsqueda Vectorial (Retrieve)
# Extraemos los documentos candidatos de la base de datos (ChromaDB)
evidencias, _, _ = pipeline.retrieve(query_demo, 2)
print(f"\n🔍 Fase 1 - Recuperación Exitosa: Se encontraron {len(evidencias)} documentos candidatos.")
# Paso 3: Construcción del Contexto (Build Context)
contexto = pipeline.build_context(evidencias)
print(f"\n📦 Fase 2 - Contexto estructurado (Vista truncada para informe):")
print("-" * 50)
# Imprimimos solo los primeros 400 caracteres para evidenciar la estructura sin saturar la hoja
print(contexto[:400] + "\n\n... [Contexto truncado por longitud] ...")
print("-" * 50)
# Paso 4: Generación con LLM (Generate Answer)
print("\n🤖 Fase 3 - Generación de respuesta (Gemini):")
if pipeline.gemini_client:
    respuesta = pipeline.generate_answer(query_demo, contexto)
    print(respuesta)
else:
    print("⚠️ GEMINI_API_KEY no configurada. (Mostrando simulación basada en el contexto).")
    print(f"Según los productos encontrados, contamos con opciones destacadas como el producto [1] que es ideal para {query_demo}.")

Sistema RAG (Literal d)
👤 Consulta del usuario: 'reading sunglasses for women'

🔍 Fase 1 - Recuperación Exitosa: Se encontraron 2 documentos candidatos.

📦 Fase 2 - Contexto estructurado (Vista truncada para informe):
--------------------------------------------------
[1] Producto: EYEGUARD Outdoor Reading Sunglasses Elegant Womens Readers Glasses-Not Bifocals …
    Descripción: EYEGUARD Outdoor Reading Sunglasses Elegant Womens Readers Glasses-Not Bifocals … . EYEGUARD . Demi . Light weight PC frame with UV400 protection acrylic anti-scratched lens. Metal hinge and Dura-Tight Screws Lens width:55mm,Nose brdige width:16mm,Temple length:132 mm In addition to fas

... [Contexto truncado por longitud] ...
--------------------------------------------------

🤖 Fase 3 - Generación de respuesta (Gemini):
Aquí tienes un producto que coincide con tu búsqueda de gafas de sol de lectura para mujer:

*   **EYEGUARD Outdoor Reading Sunglasses Elegant Womens Readers Glasses-Not Bifocals …**
    *   

**e) Interfaz conversacional:** La trazabilidad en Streamlit se logra empaquetando los resultados en objetos `RetrievedDocument`, los cuales conservan la metadata (título, métrica y ruta física) necesaria para renderizar las tarjetas de evidencia visual[cite: 3].

In [16]:
print("▶ VERIFICACIÓN PARA INTERFAZ WEB (Literal e)")
print("Extrayendo datos de la primera evidencia recuperada para simular el renderizado en Streamlit:\n")

if evidencias:
    doc = evidencias[0]
    print(f"📌 Título a mostrar (st.markdown): {doc.product_title[:60]}...")
    print(f"📊 Similitud Semántica (st.metric): {doc.score:.4f}")
    print(f"🖼️ Ruta física de la imagen local:  {doc.image_path}")
    
    # Simulación de la lógica de renderizado visual que usaría Streamlit
    print("\n🖥️ Lógica de visualización en la interfaz:")
    
    # Verificamos que la ruta exista, no sea nula y no sea una cadena vacía
    if pd.notna(doc.image_path) and doc.image_path.strip() != "" and os.path.exists(doc.image_path):
        print("✅ Resultado: Se llamará a `st.image(doc.image_path)` renderizando el archivo local exitosamente.")
    else:
        print("⚠️ Resultado: Imagen no disponible en disco. Se llamará a `st.image('placeholder.png')` como respaldo visual.")
else:
    print("No hay evidencias para evaluar.")

▶ VERIFICACIÓN PARA INTERFAZ WEB (Literal e)
Extrayendo datos de la primera evidencia recuperada para simular el renderizado en Streamlit:

📌 Título a mostrar (st.markdown): EYEGUARD Outdoor Reading Sunglasses Elegant Womens Readers G...
📊 Similitud Semántica (st.metric): 0.7346
🖼️ Ruta física de la imagen local:  C:\Users\mark_\Documents\ProyectoIIB-IR\ProyectoIIB\data\corpus\images\B01MTVBGBG.jpg

🖥️ Lógica de visualización en la interfaz:
✅ Resultado: Se llamará a `st.image(doc.image_path)` renderizando el archivo local exitosamente.


### 4. Resultados Experimentales y Evaluación (Top-k)
Para aislar la capacidad analítica del motor vectorial evaluamos las siguientes métricas:
* **Precision@k (Exactitud):** Relevantes recuperados en el Top-k / $k$.
* **Recall@k (Exhaustividad):** Relevantes recuperados en el Top-k / Total de Relevantes en el corpus.
* **NDCG@k (Calidad):** Penaliza logarítmicamente a los documentos relevantes que aparecen en posiciones bajas.

In [17]:
print("▶ CARGA DEL GROUND TRUTH Y EVALUACIÓN PASO A PASO\n")
# 1. Cargamos las consultas y los juicios de relevancia (qrels) desde disco
queries_path = Path("data/qrels/queries.json")
qrels_path = Path("data/qrels/qrels.json")

with open(queries_path, "r", encoding="utf-8") as f:
    queries_dict = json.load(f)

with open(qrels_path, "r", encoding="utf-8") as f:
    qrels_dict = json.load(f)
# 2. Seleccionamos una consulta dinámicamente (tomamos la primera disponible)
query_id_eval = list(queries_dict.keys())[0]
query_text_eval = queries_dict[query_id_eval]
print(f"👤 Consulta Seleccionada:")
print(f"   ID: {query_id_eval}")
print(f"   Texto: '{query_text_eval}'\n")
# 3. Extraemos los documentos relevantes (Ground Truth) para esta consulta
# El formato estándar de qrels es {query_id: {product_id: score}}
# Consideramos relevantes los productos con un score de relevancia > 0 (Exact o Substitute)
qrels_consulta = qrels_dict.get(query_id_eval, {})
relevantes_gt = {prod_id for prod_id, score in qrels_consulta.items() if score > 0}
print(f"📦 Total de documentos realmente relevantes en el corpus: {len(relevantes_gt)}")
print(f"   IDs Relevantes: {relevantes_gt}\n")
# 4. Ejecutamos la búsqueda y mostramos resultados en una sola línea
k_eval = 5
candidatos, _, _ = pipeline.retrieve(query_text_eval, top_k=k_eval)
candidatos_ids = [doc.doc_id for doc in candidatos]
resultados = [f"Pos {i}: {'✅' if doc_id in relevantes_gt else '❌'}" for i, doc_id in enumerate(candidatos_ids, 1)]
print(f"🔍 Top-{k_eval} Recuperados: " + " | ".join(resultados))
print(f"▶ CÁLCULO DE MÉTRICAS PARA Top-{k_eval}\n")
# A. Cálculo de Precision@k
true_positives = sum(1 for doc_id in candidatos_ids if doc_id in relevantes_gt)
precision_k = true_positives / k_eval
print(f"🎯 Precision@{k_eval}: {true_positives} aciertos / {k_eval} recuperados = {precision_k:.3f}")
# B. Cálculo de Recall@k
recall_k = true_positives / len(relevantes_gt) if len(relevantes_gt) > 0 else 0
print(f"🧲 Recall@{k_eval}: {true_positives} aciertos / {len(relevantes_gt)} relevantes totales = {recall_k:.3f}")
# C. Cálculo de NDCG@k
# Generamos el vector de relevancia binaria para los resultados devueltos [1, 0, 1...]
relevances = [1 if doc_id in relevantes_gt else 0 for doc_id in candidatos_ids]
# Calculamos DCG (Descuento Acumulado)
dcg = sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances))

# Calculamos IDCG (Ideal DCG - ordenando los relevantes primero)
ideal_relevances = sorted(relevances, reverse=True)
idcg = sum(rel / math.log2(idx + 2) for idx, rel in enumerate(ideal_relevances))

ndcg_k = dcg / idcg if idcg > 0 else 0
print(f"📈 NDCG@{k_eval}: DCG ({dcg:.3f}) / IDCG ({idcg:.3f}) = {ndcg_k:.3f}")

▶ CARGA DEL GROUND TRUTH Y EVALUACIÓN PASO A PASO

👤 Consulta Seleccionada:
   ID: 978
   Texto: '1.25 sunglasses for women'

📦 Total de documentos realmente relevantes en el corpus: 9
   IDs Relevantes: {'doc_B07RSVC23N', 'doc_B08C2FTNJ8', 'doc_B01CZN30J2', 'doc_B07215KNBH', 'doc_B01HHB9T4M', 'doc_B07MN1GKJ5', 'doc_B07DDDW2QV', 'doc_B01MTVBGBG', 'doc_B016J7E3PA'}

🔍 Top-5 Recuperados: Pos 1: ❌ | Pos 2: ❌ | Pos 3: ✅ | Pos 4: ❌ | Pos 5: ❌
▶ CÁLCULO DE MÉTRICAS PARA Top-5

🎯 Precision@5: 1 aciertos / 5 recuperados = 0.200
🧲 Recall@5: 1 aciertos / 9 relevantes totales = 0.111
📈 NDCG@5: DCG (0.500) / IDCG (1.000) = 0.500


### 5. Consolidación del Rendimiento Global (Macro-average)

In [18]:
print("▶ CARGA DE RESULTADOS AGREGADOS (Literal f)")
eval_path = Path("data/eval_results/evaluation_results.csv")

if eval_path.exists():
    df_results = pd.read_csv(eval_path)
    df_promedio = df_results[df_results['query_id'] == 'PROMEDIO'].copy()
    
    print("\n📊 Rendimiento Promedio (Macro-average) del motor Multimodal:")
    cols_metricas = [c for c in df_promedio.columns if c not in ('query_id', 'query')]
    display(df_promedio[cols_metricas].style.format("{:.3f}"))
else:
    print(f"⚠️ Reporte no encontrado en {eval_path}. Debe ejecutar el script de evaluación global previamente.")

▶ CARGA DE RESULTADOS AGREGADOS (Literal f)

📊 Rendimiento Promedio (Macro-average) del motor Multimodal:


,precision@3,recall@3,ndcg@3,precision@5,recall@5,ndcg@5,precision@10,recall@10,ndcg@10
39,0.359,0.217,0.382,0.338,0.327,0.390,0.241,0.443,0.416


**Análisis de las Métricas (Literal f):**
* **Trade-off Precision-Recall:** Para $k=3$, el sistema prioriza la exactitud (precisión 35.9%) sacrificando exhaustividad (21.7%)[cite: 3]. En $k=10$, el Recall sube a 44.3% (ideal para dar contexto al LLM), aunque la precisión baja al 24.1% por la introducción de ruido[cite: 3].
* **Calidad del Ranking (NDCG):** Crece de forma estable desde 0.382 ($k=3$) a 0.416 ($k=10$). Esto confirma que la similitud coseno sobre embeddings multimodales de CLIP alinea correctamente la semántica, posicionando los ítems útiles en los primeros lugares[cite: 3].

### 6. Funcionalidades de Excelencia Implementadas
1. **Re-ranking:** Cross-encoder que re-evalúa los resultados de ChromaDB para una mejor alineación semántica fina[cite: 3].
2. **Query Expansion:** Inyección de variantes semánticas y unificación de rankings mediante Reciprocal Rank Fusion (RRF)[cite: 3].
3. **Relevance Feedback:** Botones interactivos (👍/👎) en Streamlit que aplican el algoritmo de Rocchio para ajustar el vector de búsqueda[cite: 3].
4. **Memoria Conversacional:** Historial de turnos utilizado para reformular automáticamente preguntas anafóricas[cite: 3].